In [1]:
using CSV
using DataFrames
using Dates

In [2]:
ENV["COLUMNS"] = 1000;

In [3]:
forecast_date = "2021-01-24";
forecast_date_ = replace(forecast_date, "-" => "_");

In [4]:
forecast_start_date = Date(2021, 01, 21);

In [5]:
patienttypes = [:total, :icu, :acute];
forecasttypes = [:admitted, :active];

In [6]:
d = DataFrame(CSV.File("../data/forecasts-$(forecast_date)/jhhs_forecast_active_total_$(forecast_date_).csv"))
scenarios = unique([string(split(n, "_")[2]) for n in names(d) if n != "date"])
@show scenarios;

scenarios = ["optimistic", "moderate"]


In [7]:
function load_forecast(forecasttype, patienttype)
    d = DataFrame(CSV.File("../data/forecasts-$(forecast_date)/jhhs_forecast_$(forecasttype)_$(patienttype)_$(forecast_date_).csv"))
    d = stack(d, Not(:date))
    d.hospital = map(x -> string(split(x, "_")[1]), d.variable)
    d.scenario = map(x -> string(split(x, "_")[2]), d.variable)
    d.std = map(x -> (count("_", x) == 2) && (split(x, "_")[3] == "std"), d.variable)
    d = unstack(d, [:scenario, :hospital, :date], :std, :value)
    rename!(d, "false" => "$forecasttype", "true" => "$(forecasttype)_std")
    insertcols!(d, 1, :patienttype => fill(string(patienttype), nrow(d)))
    sort!(d, [:scenario, :hospital, :date])
    return d
end;

In [8]:
dfs = DataFrame[]
for pt in patienttypes
    dfs_ = DataFrame[]
    for ft in forecasttypes
        push!(dfs_, load_forecast(ft, pt))
    end
    df = outerjoin(dfs_..., on=[:scenario, :hospital, :date, :patienttype])
    push!(dfs, df)
end
data = vcat(dfs...);

In [9]:
filter!(row -> row.date >= forecast_start_date, data);
filter!(row -> row.admitted + row.active != 0, data);

In [10]:
data |> CSV.write("../data/longterm-$(forecast_date).csv")

"../data/longterm-2021-01-24.csv"